### 1. Cài đặt thư viện và Khởi tạo Seed

In [30]:
!pip install transformers datasets accelerate evaluate rouge_score -q

import torch
import pandas as pd
import numpy as np
import re
import ast
import random
import os
import string
import unicodedata
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, T5Tokenizer, T5ForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, Dataset
from tqdm.notebook import tqdm
import evaluate

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

### 2. Tải Mô hình ViHateT5-base-HSD

In [31]:
model_name = "tarudesu/ViHateT5-base-HSD"
print(f"Đang tải mô hình {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Hoàn tất! Mô hình đang chạy trên: {device}")

Đang tải mô hình tarudesu/ViHateT5-base-HSD...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Hoàn tất! Mô hình đang chạy trên: cuda


### 3. Các hàm Đánh giá và Xử lý nhãn

In [32]:
def generate_predictions(texts, prefix, batch_size=32, max_length=128):
    predictions = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"Dự đoán [{prefix.strip()}]"):
        batch_texts = [str(text).lower() for text in texts[i:i+batch_size]]
        inputs = [prefix + text for text in batch_texts]
        
        inputs_tokenized = tokenizer(
            inputs, padding=True, truncation=True, 
            max_length=max_length, return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs_tokenized, max_length=max_length, do_sample=False)
            
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([p.strip() for p in preds])
    return predictions

def evaluate_classification(true_labels, pred_labels):
    true_labels_norm = [str(l).strip().upper() for l in true_labels]
    pred_labels_norm = [str(p).strip().upper() for p in pred_labels]
    
    acc = accuracy_score(true_labels_norm, pred_labels_norm)
    wf1 = f1_score(true_labels_norm, pred_labels_norm, average="weighted", zero_division=0)
    mf1 = f1_score(true_labels_norm, pred_labels_norm, average="macro", zero_division=0)
    return acc, wf1, mf1

def process1_extract_spans(text):
    parts = re.split(r'\[\s*hate\s*\]', text, flags=re.IGNORECASE)
    spans = [parts[i].strip() for i in range(1, len(parts), 2)]
    return [s for s in spans if s]

def process2_to_binary_seq(original_text, spans):
    tokens = str(original_text).split()
    binary_seq = np.zeros(len(tokens), dtype=int)
    
    def clean_word(w):
        w = unicodedata.normalize('NFC', str(w))
        return w.lower().strip(string.punctuation)
        
    tokens_clean = [clean_word(t) for t in tokens]
    
    for span in spans:
        span_tokens = [clean_word(t) for t in span.split()]
        span_len = len(span_tokens)
        if span_len == 0: continue
        for i in range(len(tokens_clean) - span_len + 1):
            if tokens_clean[i:i+span_len] == span_tokens:
                binary_seq[i:i+span_len] = 1
    return binary_seq.tolist()

def gt_char_spans_to_binary_seq(original_text, index_spans_str):
    original_text = str(original_text)
    tokens = original_text.split()
    binary_seq = np.zeros(len(tokens), dtype=int)
    
    try:
        char_indices = set(ast.literal_eval(str(index_spans_str)))
    except:
        char_indices = set()
        
    if not char_indices: return binary_seq.tolist()
        
    token_idx, in_token = 0, False
    for char_idx, char in enumerate(original_text):
        if not char.isspace(): 
            if not in_token: in_token = True
            if char_idx in char_indices and token_idx < len(tokens):
                binary_seq[token_idx] = 1
        else: 
            if in_token:
                in_token = False
                token_idx += 1 
    return binary_seq.tolist()

def evaluate_vihos(original_texts, gt_index_spans, pred_texts):
    vihos_acc, vihos_wf1, vihos_mf1 = [], [], []
    for orig, gt_spans_str, pred in zip(original_texts, gt_index_spans, pred_texts):
        orig_str_norm = unicodedata.normalize('NFC', str(orig).lower())
        pred_str_norm = unicodedata.normalize('NFC', str(pred).lower())
        
        substrings = []
        s_idx = pred_str_norm.find('[hate]')
        while s_idx != -1:
            e_idx = pred_str_norm.find('[hate]', s_idx + 6)
            if e_idx != -1:
                substrings.append(pred_str_norm[s_idx + 6 : e_idx])
                s_idx = pred_str_norm.find('[hate]', e_idx + 6)
            else:
                break
                
        pred_indices = []
        for sub in substrings:
            if not sub: continue
            idx = orig_str_norm.find(sub)
            while idx != -1:
                pred_indices.extend(list(range(idx, idx + len(sub))))
                idx = orig_str_norm.find(sub, idx + 1)
        pred_indices = set(pred_indices)
        
        try:
            gt_indices = set(ast.literal_eval(str(gt_spans_str)))
        except:
            gt_indices = set()
            
        length = len(str(orig))
        y_true = [1 if i in gt_indices else 0 for i in range(length)]
        y_pred = [1 if i in pred_indices else 0 for i in range(length)]
        
        vihos_acc.append(accuracy_score(y_true, y_pred))
        vihos_wf1.append(f1_score(y_true, y_pred, average='weighted', zero_division=0))
        vihos_mf1.append(f1_score(y_true, y_pred, average='macro', zero_division=0))
        
    return np.mean(vihos_acc), np.mean(vihos_wf1), np.mean(vihos_mf1)

### 4. Đọc dữ liệu (ViHSD, ViCTSD, ViHOS)

In [33]:
print("Đang đọc dữ liệu...")

# 1. ViHSD 
df_vihsd = pd.read_csv("/kaggle/input/datasets/khanhduynguyenhuu/vihsd-data/vihsd/test.csv")
vihsd_label_map = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}
vihsd_data = {
    'texts': df_vihsd['free_text'].fillna("").astype(str).tolist(),
    'labels': [vihsd_label_map[int(l)] for l in df_vihsd['label_id']]
}

# 2. ViCTSD
victsd_dataset = load_dataset("tarudesu/ViCTSD", split="test")
def map_victsd_label(val): return "TOXIC" if int(val) == 1 else "NONE"
victsd_data = {
    'texts': victsd_dataset['Comment'],
    'labels': [map_victsd_label(l) for l in victsd_dataset['Toxicity']]
}

# 3. ViHOS
df_vihos = pd.read_csv("/kaggle/input/datasets/khanhduynguyenhuu/vihos-data-2/test.csv")
vihos_data = {
    'texts': df_vihos['content'].fillna("").astype(str).tolist(),
    'labels': df_vihos['index_spans'].astype(str).tolist()
}
print("Dữ liệu đã sẵn sàng!")

Đang đọc dữ liệu...
Dữ liệu đã sẵn sàng!


### 5. Thực hiện lại Bảng 3 (Đánh giá trên 3 tập dữ liệu)

In [34]:
print("--- 1. BẮT ĐẦU ĐÁNH GIÁ ViHSD ---")
vihsd_preds = generate_predictions(vihsd_data['texts'], "hate-speech-detection: ")
v_acc, v_wf1, v_mf1 = evaluate_classification(vihsd_data['labels'], vihsd_preds)
print(f"ViHSD Kết quả -> Acc: {v_acc:.4f} | WF1: {v_wf1:.4f} | MF1: {v_mf1:.4f}\n")

print("--- 2. BẮT ĐẦU ĐÁNH GIÁ ViCTSD ---")
victsd_preds = generate_predictions(victsd_data['texts'], "toxic-speech-detection: ")
c_acc, c_wf1, c_mf1 = evaluate_classification(victsd_data['labels'], victsd_preds)
print(f"ViCTSD Kết quả -> Acc: {c_acc:.4f} | WF1: {c_wf1:.4f} | MF1: {c_mf1:.4f}\n")

print("--- 3. BẮT ĐẦU ĐÁNH GIÁ ViHOS ---")
vihos_preds = generate_predictions(vihos_data['texts'], "hate-spans-detection: ", max_length=256)
h_acc, h_wf1, h_mf1 = evaluate_vihos(vihos_data['texts'], vihos_data['labels'], vihos_preds)
print(f"ViHOS Kết quả -> Acc: {h_acc:.4f} | WF1: {h_wf1:.4f} | MF1: {h_mf1:.4f}\n")

avg_mf1 = (v_mf1 + c_mf1 + h_mf1) / 3
print("====================================")
print(f"ĐIỂM MF1 TRUNG BÌNH: {avg_mf1:.4f}")
print("====================================")

--- 1. BẮT ĐẦU ĐÁNH GIÁ ViHSD ---


Dự đoán [hate-speech-detection:]:   0%|          | 0/209 [00:00<?, ?it/s]

ViHSD Kết quả -> Acc: 0.8862 | WF1: 0.8822 | MF1: 0.6836

--- 2. BẮT ĐẦU ĐÁNH GIÁ ViCTSD ---


Dự đoán [toxic-speech-detection:]:   0%|          | 0/32 [00:00<?, ?it/s]

ViCTSD Kết quả -> Acc: 0.9080 | WF1: 0.8982 | MF1: 0.7163

--- 3. BẮT ĐẦU ĐÁNH GIÁ ViHOS ---


Dự đoán [hate-spans-detection:]:   0%|          | 0/35 [00:00<?, ?it/s]

ViHOS Kết quả -> Acc: 0.9100 | WF1: 0.9014 | MF1: 0.8636

ĐIỂM MF1 TRUNG BÌNH: 0.7545


In [42]:
print("--- CHẠY THỬ MÔ HÌNH VỚI 3 VÍ DỤ ---\n")

# 1. Task ViHSD (Nhận diện Bình luận Thù địch)
sample_vihsd = "Đúng là đồ vô văn hóa, cút đi cho khuất mắt"
gt_vihsd = "HATE"
pred_vihsd = generate_predictions([sample_vihsd], "hate-speech-detection: ")[0]
pred_vihsd = "HATE" if pred_vihsd.upper() == "OFFENSIVE" else pred_vihsd.upper()
print(f"Task 1 (ViHSD)\nCâu gốc: {sample_vihsd}")
print(f"-> Đáp án chuẩn (GT): {gt_vihsd}")
print(f"-> Mô hình dự đoán:        {pred_vihsd}\n")

# 2. Task ViCTSD (Nhận diện Bình luận Độc hại)
sample_victsd = "Đóng phim dở tệ mà cứ thích ra vẻ, nhìn là thấy ghét"
gt_victsd = "TOXIC"
pred_victsd = generate_predictions([sample_victsd], "toxic-speech-detection: ")[0]
print(f"Task 2 (ViCTSD)\nCâu gốc: {sample_victsd}")
print(f"-> Đáp án chuẩn (GT): {gt_victsd}")
print(f"-> Mô hình dự đoán:        {pred_victsd.upper()}\n")

# 3. Task ViHOS (Trích xuất cụm từ thù địch)
sample_vihos = "Mấy thằng não bò này cãi cùn ghê thật, lũ rảnh háng"
gt_vihos = "['thằng não bò', 'lũ rảnh háng']"
pred_vihos = generate_predictions([sample_vihos], "hate-spans-detection: ")[0]
spans_vihos = process1_extract_spans(pred_vihos)
print(f"Task 3 (ViHOS)\nCâu gốc: {sample_vihos}")
print(f"-> Cụm từ vi phạm (GT): {gt_vihos}")
print(f"-> Mô hình bóc tách:         {spans_vihos}\n")


--- CHẠY THỬ MÔ HÌNH VỚI 3 VÍ DỤ ---



Dự đoán [hate-speech-detection:]:   0%|          | 0/1 [00:00<?, ?it/s]

Task 1 (ViHSD)
Câu gốc: Đúng là đồ vô văn hóa, cút đi cho khuất mắt
-> Đáp án chuẩn (GT): HATE
-> Mô hình dự đoán:        HATE



Dự đoán [toxic-speech-detection:]:   0%|          | 0/1 [00:00<?, ?it/s]

Task 2 (ViCTSD)
Câu gốc: Đóng phim dở tệ mà cứ thích ra vẻ, nhìn là thấy ghét
-> Đáp án chuẩn (GT): TOXIC
-> Mô hình dự đoán:        TOXIC



Dự đoán [hate-spans-detection:]:   0%|          | 0/1 [00:00<?, ?it/s]

Task 3 (ViHOS)
Câu gốc: Mấy thằng não bò này cãi cùn ghê thật, lũ rảnh háng
-> Cụm từ vi phạm (GT): ['thằng não bò', 'lũ rảnh háng']
-> Mô hình bóc tách:         ['thằng não bò', 'lũ rảnh háng']



### 6. Tạo Mini Dataset cho bài toán QA (Tùy chọn)

In [35]:
# Code tạo dataset được comment lại để tham khảo
# df_vihsd_full = pd.DataFrame(vihsd_data)
# df_clean = df_vihsd_full[df_vihsd_full['labels'].str.upper() == 'CLEAN'].copy()
# df_hate = df_vihsd_full[df_vihsd_full['labels'].str.upper() == 'HATE'].copy()

# sample_clean = df_clean.sample(n=500, random_state=42)
# sample_hate = df_hate.sample(n=500, random_state=42)

# sample_clean['input_text'] = sample_clean['texts'].apply(lambda x: f"hate-speech-qa: Ngữ cảnh: '{x}' Câu hỏi: Tại sao câu này độc hại?")
# sample_hate['input_text'] = sample_hate['texts'].apply(lambda x: f"hate-speech-qa: Ngữ cảnh: '{x}' Câu hỏi: Tại sao câu này độc hại?")

# sample_clean['target_reason'] = "Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm."
# sample_hate['target_reason'] = ""

# sample_clean[['input_text', 'target_reason']].to_csv('Clean_data_500.csv', index=False, encoding='utf-8-sig')
# sample_hate[['input_text', 'target_reason']].to_csv('Hate_data_500_blank.csv', index=False, encoding='utf-8-sig')
# print("Đã tạo xong file Dataset mẫu!")

### 7. Fine-Tune mô hình cho bài toán Giải thích (QA Task)

In [43]:
file_path = '/kaggle/input/datasets/khanhduynguyenhuu/ground-truth-2/vihsd_qa_gemini_GT_Final.csv'
try:
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['input_text', 'target_reason'])

    dataset = Dataset.from_pandas(df)
    dataset = dataset.train_test_split(test_size=0.2, seed=42)

    qa_tokenizer = T5Tokenizer.from_pretrained("tarudesu/ViHateT5-base-HSD")
    qa_model = T5ForConditionalGeneration.from_pretrained("tarudesu/ViHateT5-base-HSD")

    def preprocess_function(examples):
        model_inputs = qa_tokenizer(
            examples["input_text"], max_length=256, truncation=True, padding="max_length"
        )
        labels = qa_tokenizer(
            text_target=examples["target_reason"], max_length=128, truncation=True, padding="max_length"
        )
        labels["input_ids"] = [
            [(l if l != qa_tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_datasets = dataset.map(preprocess_function, batched=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir="./hate_qa_model_checkpoints",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=1,
        num_train_epochs=5,
        predict_with_generate=True,
        fp16=True,
        seed=42,             # Cố định seed
        data_seed=42,        # Cố định data seed
        load_best_model_at_end=True, # Lấy model tốt nhất
    )

    data_collator = DataCollatorForSeq2Seq(qa_tokenizer, model=qa_model)

    trainer = Seq2SeqTrainer(
        model=qa_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        data_collator=data_collator,
        processing_class=qa_tokenizer,
    )

    print("Bắt đầu quá trình Huấn luyện (Fine-Tuning)...")
    trainer.train() # Bỏ comment để chạy huấn luyện
    
    trainer.save_model("./hate_qa_model_final")
    qa_tokenizer.save_pretrained("./hate_qa_model_final")
    print("Huấn luyện hoàn tất!")

except FileNotFoundError:
    print(f"Không tìm thấy file {file_path}. Vui lòng cập nhật lại đường dẫn.")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Bắt đầu quá trình Huấn luyện (Fine-Tuning)...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Huấn luyện hoàn tất!


### 8. Đánh giá kết quả sinh văn bản bằng ROUGE

In [44]:
rouge = evaluate.load('rouge')

# device = "cuda" if torch.cuda.is_available() else "cpu"
# final_model_path = "./hate_qa_model_final"
# qa_model = T5ForConditionalGeneration.from_pretrained(final_model_path).to(device)
# qa_tokenizer = T5Tokenizer.from_pretrained(final_model_path)

try:
    test_data = dataset['test']
    predictions = []
    references = []

    print("Đang đánh giá mô hình trên tập Test...")
    for i in range(len(test_data)):
        input_text = test_data[i]['input_text']
        ground_truth = test_data[i]['target_reason']
        
        inputs = qa_tokenizer(input_text, return_tensors="pt", max_length=256, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = qa_model.generate(
                **inputs, 
                max_length=128, 
                num_beams=4,
                repetition_penalty=1.5
            )
            
        pred_text = qa_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if pred_text.startswith("âu "):
            pred_text = "C" + pred_text
        elif pred_text.startswith("ử dụng "):
            pred_text = "S" + pred_text
        elif pred_text.startswith("ừ "):
            pred_text = "T" + pred_text
        elif len(pred_text) > 0:
            pred_text = pred_text[0].upper() + pred_text[1:]
        
        predictions.append(pred_text)
        references.append(ground_truth)

    # Chấm điểm toàn bộ
    results = rouge.compute(predictions=predictions, references=references)
    print("\n==================================================")
    print("KẾT QUẢ ROUGE (TOÀN BỘ TẬP TEST)")
    print("==================================================")
    print(f"ROUGE-1: {results['rouge1'] * 100:.2f}%")
    print(f"ROUGE-2: {results['rouge2'] * 100:.2f}%")
    print(f"ROUGE-L: {results['rougeL'] * 100:.2f}%")

    # Cô lập nhóm HATE để đánh giá 
    hate_preds = []
    hate_refs = []
    for pred, ref in zip(predictions, references):
        if ref != "Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm.":
            hate_preds.append(pred)
            hate_refs.append(ref)

    hate_results = rouge.compute(predictions=hate_preds, references=hate_refs)
    print("\n==================================================")
    print("KẾT QUẢ ROUGE (CHỈ NHÓM HATE)")
    print("==================================================")
    print(f"ROUGE-1: {hate_results['rouge1'] * 100:.2f}%")
    print(f"ROUGE-2: {hate_results['rouge2'] * 100:.2f}%")
    print(f"ROUGE-L: {hate_results['rougeL'] * 100:.2f}%")

except NameError:
    print("Chạy quá trình huấn luyện ở trên trước khi đánh giá.")

Đang đánh giá mô hình trên tập Test...

KẾT QUẢ ROUGE (TOÀN BỘ TẬP TEST)
ROUGE-1: 78.79%
ROUGE-2: 61.57%
ROUGE-L: 68.42%

KẾT QUẢ ROUGE (CHỈ NHÓM HATE)
ROUGE-1: 55.86%
ROUGE-2: 20.08%
ROUGE-L: 33.99%


In [45]:
import random

print("--- XEM THỬ 5 VÍ DỤ NGẪU NHIÊN TỪ TẬP TEST ---\n")
try:
    # Chọn ngẫu nhiên 5 câu từ tập Test
    sample_indices = random.sample(range(len(test_data)), 5)
    
    for i, idx in enumerate(sample_indices):
        input_text = test_data[idx]['input_text']
        # Cắt gọt cái Prefix 
        clean_input = input_text.replace("hate-speech-qa: Ngữ cảnh: '", "").replace("' Câu hỏi: Tại sao câu này độc hại?", "")
        
        print(f"VÍ DỤ {i+1}:")
        print(f"Câu gốc: {clean_input}")
        print(f"-> Đáp án chuẩn (GT): {references[idx]}")
        print(f"-> Mô hình giải thích:     {predictions[idx]}\n")
        print("-"*80 + "\n")
        
except NameError:
    print("Lỗi: Chạy Cell đánh giá ROUGE ở trên trước để lấy kết quả dự đoán (predictions).")


--- XEM THỬ 5 VÍ DỤ NGẪU NHIÊN TỪ TẬP TEST ---

VÍ DỤ 1:
Câu gốc: Vậy mà vẫn thua Thi Thi ak,
-> Đáp án chuẩn (GT): Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm.
-> Mô hình giải thích:     Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm.

--------------------------------------------------------------------------------

VÍ DỤ 2:
Câu gốc: Vải của now tốt nè
-> Đáp án chuẩn (GT): Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm.
-> Mô hình giải thích:     Câu bình luận này là bình thường, không chứa ngôn từ thù địch hay xúc phạm.

--------------------------------------------------------------------------------

VÍ DỤ 3:
Câu gốc: Chậm chậm clg mấy con bò rảnh háng này
-> Đáp án chuẩn (GT): Sử dụng từ ngữ tục tĩu ('clg'), xúc phạm nhân phẩm ('mấy con bò') và miệt thị giới tính ('rảnh háng') để hạ thấp người khác.
-> Mô hình giải thích:     Sử dụng từ ngữ tục tĩu, xúc phạm như 'con bò', 'rảnh háng' để xúc phạm